## XGBoost TRaining using spaCy

In [1]:
# Necessary libraries for the full process (preprocessing, vectorization, training, evaluation)
import os 
import pandas as pd
import spacy # Essential for NLP preprocessing
import re # Essential for URL/special character removal
from xgboost import XGBClassifier # For the final model training
from sklearn.model_selection import train_test_split # For splitting data
from sklearn.metrics import f1_score # For evaluating the model
from sklearn.feature_extraction.text import TfidfVectorizer # Kept for potential use
from sklearn.decomposition import TruncatedSVD # Kept for potential use (Dimensionality Reduction)
from spacy.tokens import Doc
from typing import List

RND = 42 # Random state for reproducibility

In [2]:
# Load the expanded dataset
DATA_PATH = '../resources/dataset/synonym_youtoxic_english_1000.csv'
assert os.path.exists(DATA_PATH), f"Data file not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
print('Loaded', DATA_PATH, 'shape=', df.shape)
df.head()

Loaded ../resources/dataset/synonym_youtoxic_english_1000.csv shape= (3751, 16)


,Unnamed: 0,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement embody not train to shoot to a...,True,True,False,False,False,False,False,False,False,False,False,False
3,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement make up not trained to shoot t...,True,True,False,False,False,False,False,False,False,False,False,False
4,1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Legal philosophy enforcement is not trained to...,True,True,False,False,False,False,False,False,False,False,False,False


In [3]:
# Identify label columns (common names used in the original notebook)
target_cols = [
    'IsToxic', 'IsAbusive', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist'
]
# Keep only the labels that are present in the dataset
present_targets = [c for c in target_cols if c in df.columns]
if len(present_targets) == 0:
    raise ValueError("No expected target columns found in the uploaded CSV. Please ensure the file contains at least one of: " + ','.join(target_cols))
print('Using target columns:', present_targets)

Using target columns: ['IsToxic', 'IsAbusive', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist']


In [4]:
# Safe text column detection
text_col = None
for candidate in ['text','Text','comment_text','comment','commentText']:
    if candidate in df.columns:
        text_col = candidate
        break
if text_col is None:
    raise ValueError("No text column found in data (expected one of: text, Text, comment_text, comment, commentText).")
print('Using text column:', text_col)
df[text_col] = df[text_col].astype(str)

Using text column: Text


## Install & load spaCy and models

In [5]:

# Load the spaCy model (small model for speed; use 'en_core_web_md' or 'en_core_web_lg' for better vectors/NER)
nlp = spacy.load('en_core_web_sm', disable=['parser'])
# We disabled the dependency parser for speed. We'll keep tagger, lemmatizer and ner if needed.
# If you want NER, enable it: spacy.load('en_core_web_sm') or keep 'ner' in components.

# Optional: extend stop words (for toxic words you might want to *keep* some tokens normally in stopwords, so review before removing)
# nlp.Defaults.stop_words.add('u')  # example

## Remove URLs

In [8]:
# Regex pattern to match URLs (including http, https, www, or text that looks like a domain)
URL_PATTERN = r'https?://\S+|www\.\S+|\S+\.(?:com|org|net|edu|gov|io|co|ai|ly)\S*'

def remove_and_count_urls(text: str) -> str:
    """Removes URLs from a text string and updates a global counter."""
    global url_count
    urls_found = re.findall(URL_PATTERN, text)
    if urls_found:
        url_count += len(urls_found)
        # Replace the URLs with an empty string
        text = re.sub(URL_PATTERN, '', text)
    return text

# Initialize a global counter for URLs
url_count = 0

# Apply the function to the text column
df[text_col] = df[text_col].apply(remove_and_count_urls)

# --- Summary and Message ---
print('\n--- URL Removal Summary ---')
if url_count > 0:
    print(f"✅ Successfully removed **{url_count}** URLs from the dataset.")
    print(f"Dataframe shape after removal: {df.shape}")
else:
    # *** Modification: Print a specific message when no URLs are found ***
    print("✅ **No URLs were found** in the dataset.")
    print(f"Dataframe shape remains: {df.shape}")
print('---------------------------')


--- URL Removal Summary ---
✅ Successfully removed **12** URLs from the dataset.
Dataframe shape after removal: (3751, 16)
---------------------------


In [9]:
# Re-import re just in case (though it should be imported from the first cell)
import re

# Define the pattern to remove (non-word, non-whitespace, non-basic punctuation)
# This pattern removes anything not:
# \w (word characters: letters, numbers, underscore)
# \s (whitespace)
# ' " ! ? . , - (common punctuation/symbols we want to keep)
# You can customize this list of characters to keep.
SPECIAL_CHAR_PATTERN = re.compile(r'[^\w\s\'"!?.,-]')

# Initialize a global counter for special characters removed
special_char_count = 0

def remove_special_chars_and_count(text: str) -> str:
    """Removes special characters from a text string and updates a global counter."""
    global special_char_count
    
    # Temporarily find all characters that *will be removed*
    chars_to_remove = SPECIAL_CHAR_PATTERN.findall(text)
    
    # 1. Remove unusual symbols/emojis
    cleaned = SPECIAL_CHAR_PATTERN.sub(' ', text)
    
    # 2. Collapse multiple spaces and strip
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    # Update the counter based on the characters found *before* collapsing spaces
    special_char_count += len(chars_to_remove)
    
    return cleaned

# Apply the function to the text column (using the text_col variable from the previous cells)
# We assume 'text_col' and 'df' are available from the previous executions.
df[text_col] = df[text_col].apply(remove_special_chars_and_count)

# --- Summary and Message ---
print('\n--- Special Character Removal Summary ---')
if special_char_count > 0:
    print(f"✅ Successfully removed **{special_char_count}** special characters/symbols from the dataset.")
    print(f"Dataframe shape after cleaning: {df.shape}")
else:
    print("✅ **No special characters** matching the pattern were found/removed.")
    print(f"Dataframe shape remains: {df.shape}")
print('-----------------------------------------')


--- Special Character Removal Summary ---
✅ Successfully removed **1690** special characters/symbols from the dataset.
Dataframe shape after cleaning: (3751, 16)
-----------------------------------------


In [10]:
# Assuming 'df' and 'text_col' are available from the previous cells

def lowercase_text(text: str) -> str:
    """Converts the input text string to lowercase."""
    return text.lower()

# Apply the function to the text column
# This overwrites the column with the lowercase version of the text
df[text_col] = df[text_col].apply(lowercase_text)

# --- Verification and Message ---
print('\n--- Lowercasing Summary ---')
print("✅ Successfully converted all text in the column to **lowercase**.")
print('First 5 entries after lowercasing:')

# Display the head of the cleaned text column for verification
print(df[text_col].head())
print('---------------------------')


--- Lowercasing Summary ---
✅ Successfully converted all text in the column to **lowercase**.
First 5 entries after lowercasing:
0    if only people would just take a step back and...
1    law enforcement is not trained to shoot to app...
2    law enforcement embody not train to shoot to a...
3    law enforcement make up not trained to shoot t...
4    legal philosophy enforcement is not trained to...
Name: Text, dtype: object
---------------------------


In [11]:
# Necessary libraries (already imported in your first cell)
# from spacy.tokens import Doc
# from typing import List

# Assume 'df', 'text_col', and 'nlp' are available from previous cells

def tokenize_and_lemmatize(text: str) -> List[str]:
    """
    Processes text to tokenizes, lemmatizes, and removes stop words.
    
    Args:
        text: The input text string.
        
    Returns:
        A list of lemmatized, non-stop-word tokens.
    """
    # 1. Process the text using the loaded spaCy model
    # This automatically performs tokenization, POS tagging, and lemmatization
    doc = nlp(text)
    
    tokens = []
    
    # 2. Iterate through tokens and apply filtering/lemmatization
    for token in doc:
        # Check if the token is not a stop word AND is not just punctuation or whitespace
        if not token.is_stop and not token.is_punct and not token.is_space:
            # 3. Append the lowercase lemma (base form) of the word
            # .lemma_ returns the base form (e.g., 'running' -> 'run')
            tokens.append(token.lemma_.lower())
            
    return tokens

# Apply the function to the text column
# We create a new column to store the list of tokens/lemmas
# Storing lists of tokens is much more flexible than a single string
df['tokens'] = df[text_col].apply(tokenize_and_lemmatize)

# --- Verification and Message ---
print('\n--- Tokenization & Lemmatization Summary ---')
print("✅ Successfully tokenized, lemmatized, and removed stop words.")
print('Created a new column named **tokens** containing the list of lemmas.')

# Display the original text and the new tokens column for verification
print('\nFirst 5 entries showing text and tokens:')
print(df[[text_col, 'tokens']].head())
print('-----------------------------------------')


--- Tokenization & Lemmatization Summary ---
✅ Successfully tokenized, lemmatized, and removed stop words.
Created a new column named **tokens** containing the list of lemmas.

First 5 entries showing text and tokens:
                                                Text  \
0  if only people would just take a step back and...   
1  law enforcement is not trained to shoot to app...   
2  law enforcement embody not train to shoot to a...   
3  law enforcement make up not trained to shoot t...   
4  legal philosophy enforcement is not trained to...   

                                              tokens  
0  [people, step, case, people, situation, lump, ...  
1  [law, enforcement, train, shoot, apprehend, tr...  
2  [law, enforcement, embody, train, shoot, appre...  
3  [law, enforcement, train, shoot, pick, train, ...  
4  [legal, philosophy, enforcement, train, burgeo...  
-----------------------------------------


In [12]:
from collections import Counter
from itertools import chain
from typing import List

# --- Step 1: Count Token Frequencies and Identify Top 100 ---

# Define the new number of words to filter
N_WORDS_TO_FILTER = 100 

# 1. Chain all token lists into a single flat list
# This creates one long list of every single word in the dataset
all_tokens = list(chain.from_iterable(df['tokens']))

# 2. Count the frequency of each unique token
token_counts = Counter(all_tokens)

# 3. Get the N most common tokens and their counts
most_common_tokens = token_counts.most_common(N_WORDS_TO_FILTER)

# Extract just the words to create the set for filtering
high_frequency_stopwords = {word for word, count in most_common_tokens}

# --- Verification and Message for Top Words ---
print('\n--- High-Frequency Token Analysis ---')
print(f"Top {N_WORDS_TO_FILTER} most common tokens identified:")

# Print the list of the words and their exact counts
# Printing a sample of the list (e.g., the first 10) for readability
print(f"Showing the top 10 out of {N_WORDS_TO_FILTER} words:")
for word, count in most_common_tokens[:10]:
    print(f"- **{word}**: {count} occurrences")

print(f"\nThese {N_WORDS_TO_FILTER} words will be added to the filter list.")
print('-------------------------------------')



--- High-Frequency Token Analysis ---
Top 100 most common tokens identified:
Showing the top 10 out of 100 words:
- **black**: 1111 occurrences
- **people**: 832 occurrences
- **white**: 683 occurrences
- **t**: 650 occurrences
- **s**: 620 occurrences
- **police**: 561 occurrences
- **shoot**: 522 occurrences
- **like**: 511 occurrences
- **cop**: 484 occurrences
- **brown**: 431 occurrences

These 100 words will be added to the filter list.
-------------------------------------


In [13]:
# --- Step 2: Apply Frequency-Based Filtering ---

def remove_high_frequency_words(tokens: List[str]) -> List[str]:
    """
    Filters a list of tokens, removing any tokens present in the high-frequency set.
    """
    return [token for token in tokens if token not in high_frequency_stopwords]

# Apply the filtering function to the existing 'tokens' column
df['tokens'] = df['tokens'].apply(remove_high_frequency_words)

# --- Verification and Final Summary ---
print('\n--- Frequency-Based Stop Word Removal Summary (N=100) ---')
print(f"✅ Successfully filtered the **{len(high_frequency_stopwords)}** highest frequency words.")

# Display the token column again to verify removal
print('\nFirst 5 entries showing the final cleaned tokens:')
print(df[['tokens']].head())
print('--------------------------------------------------')


--- Frequency-Based Stop Word Removal Summary (N=100) ---
✅ Successfully filtered the **100** highest frequency words.

First 5 entries showing the final cleaned tokens:
                                              tokens
0  [step, case, situation, lump, mess, kind, self...
1  [enforcement, train, apprehend, train, thank, ...
2  [enforcement, embody, train, apprehend, prepar...
3  [enforcement, train, pick, train, unity, thank...
4  [legal, philosophy, enforcement, train, burgeo...
--------------------------------------------------


## 🔢 TF-IDF Vectorization Implementation

In [14]:
# Assuming 'df' and 'tokens' column are available

# 1. Join the list of tokens back into a single string per comment
df['cleaned_text'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))

print('\n--- Text Preparation for TF-IDF ---')
print("Successfully created 'cleaned_text' column from tokens.")
print('Example of cleaned text:')
print(df[['cleaned_text']].head())
print('-----------------------------------')


--- Text Preparation for TF-IDF ---
Successfully created 'cleaned_text' column from tokens.
Example of cleaned text:
                                        cleaned_text
0  step case situation lump mess kind selfish rat...
1  enforcement train apprehend train thank wilson...
2  enforcement embody train apprehend prepare fri...
3  enforcement train pick train unity thank wilso...
4  legal philosophy enforcement train burgeon for...
-----------------------------------


2. Split Data
Before fitting the TfidfVectorizer, it is crucial to split your data into training and testing sets. This ensures the vectorizer learns vocabulary only from the training data, preventing data leakage.

In [15]:
from sklearn.model_selection import train_test_split

# Assuming 'text_col' (now 'cleaned_text') and 'present_targets' are available

# Define features (X) and target (y)
X = df['cleaned_text']
# For this step, let's focus on the first target column for simplicity (e.g., 'IsToxic')
# You'll likely train separate models for each target later.
y = df[present_targets[0]]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, # Common size for test set (20%)
    random_state=RND, 
    stratify=y # Important for imbalanced classification like toxicity
)

print('\n--- Data Split Summary ---')
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Target column being used: {present_targets[0]}")
print('--------------------------')


--- Data Split Summary ---
X_train shape: (3000,)
X_test shape: (751,)
Target column being used: IsToxic
--------------------------


3. Fit and Transform TF-IDF
Now we create the vectorizer, fit it on the training data only, and then transform both the training and testing data into numerical feature matrices.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF Vectorizer
# max_df=0.8 means words appearing in more than 80% of documents are ignored (too common)
# min_df=5 means words appearing in fewer than 5 documents are ignored (too rare)
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.8,
    min_df=5,
    ngram_range=(1, 2) # Include single words (unigrams) and 2-word phrases (bigrams)
)

# Fit the vectorizer on the TRAINING data
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform the TRAINING and TEST data
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# --- TF-IDF Summary ---
print('\n--- TF-IDF Vectorization Summary ---')
print(f"Total features (vocabulary size): **{X_train_tfidf.shape[1]}**")
print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"X_test_tfidf shape: {X_test_tfidf.shape}")
print('Features successfully created for XGBoost training.')
print('--------------------------------------')


--- TF-IDF Vectorization Summary ---
Total features (vocabulary size): **3909**
X_train_tfidf shape: (3000, 3909)
X_test_tfidf shape: (751, 3909)
Features successfully created for XGBoost training.
--------------------------------------


The results look great! 🎉 Your TF-IDF vectorization successfully created 3,909 features (unique words/bigrams) and your data is properly split into training (3,000 samples) and testing (751 samples) sets.

## 🚀 Step 4: Training XGBoost with TF-IDF Features

In [17]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time

# Initialize the XGBoost Classifier
# We set 'use_label_encoder=False' and 'eval_metric' to silence warnings 
# and ensure a suitable metric for binary classification.
xgb_tfidf = XGBClassifier(
    objective='binary:logistic',  # For binary classification (IsToxic or not)
    n_estimators=100,             # Number of boosting rounds (trees)
    learning_rate=0.1,            # Step size shrinkage to prevent overfitting
    random_state=RND,
    use_label_encoder=False,
    eval_metric='logloss',        # Metric for evaluation during training
    n_jobs=-1                     # Use all available cores
)

print('Starting XGBoost training with TF-IDF features...')
start_time = time.time()

# Fit the model to the training data
xgb_tfidf.fit(X_train_tfidf, y_train)

end_time = time.time()
training_time = end_time - start_time
print(f'Training complete in {training_time:.2f} seconds.')

Starting XGBoost training with TF-IDF features...


/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [09:49:24] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete in 7.53 seconds.


2. Evaluating the Model

In [19]:
# Make predictions on the test set
y_pred_tfidf = xgb_tfidf.predict(X_test_tfidf)

# Calculate the F1-Score (often used for classification/imbalanced data)
f1_tfidf = f1_score(y_test, y_pred_tfidf)

# Generate a full classification report
report_tfidf = classification_report(y_test, y_pred_tfidf)

# --- Results Summary (TF-IDF) ---
print('\n--- Results: XGBoost with TF-IDF Features ---')
print(f"**F1-Score (IsToxic):** {f1_tfidf:.4f}")
print('\nClassification Report:')
print(report_tfidf)
print('-------------------------------------------')


--- Results: XGBoost with TF-IDF Features ---
**F1-Score (IsToxic):** 0.9352

Classification Report:
              precision    recall  f1-score   support

       False       0.95      0.19      0.31       108
        True       0.88      1.00      0.94       643

    accuracy                           0.88       751
   macro avg       0.92      0.59      0.62       751
weighted avg       0.89      0.88      0.85       751

-------------------------------------------


### ✋🏽 stop train only with (IsToxic) not valid 

Step 1: Training and Evaluation Loop (TF-IDF)

In [20]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import time

# Assuming X_train_tfidf, X_test_tfidf, RND, and present_targets are available
# X is the clean text column, y is the target column
X = df['cleaned_text']

# List to store all results for comparison later
all_tfidf_results = []
trained_tfidf_models = {}

def train_and_evaluate_target(target_col, X_data, y_data):
    """Splits data, trains XGBoost, evaluates, and reports overfitting."""
    
    # 1. Split Data (Crucial: Stratification is essential for imbalanced targets)
    y = y_data[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y, 
        test_size=0.2, 
        random_state=RND, 
        stratify=y
    )
    
    # 2. Vectorize the split data (re-using the matrix from the previous cell for X_train/X_test)
    # NOTE: Since the splits change for each target (due to stratify), 
    # we must re-vectorize or re-split the already-transformed matrix. 
    # For simplicity and to correctly match the split, we'll slice the original TF-IDF matrices.
    # However, since we defined X_train_tfidf and X_test_tfidf previously on the entire X, 
    # we need to re-run the split using the TF-IDF matrix directly for consistency.
    # We will use the original X_train_tfidf and X_test_tfidf matrices here, 
    # but acknowledge that this is an approximation since the split was tied to y_data[present_targets[0]].
    # For this loop, we MUST redefine the split to use the correct target (y) for stratification:
    
    # Redefine split based on original text (X) and current target (y)
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        X_data, y, 
        test_size=0.2, 
        random_state=RND, 
        stratify=y
    )
    
    # Re-vectorize the current split for safety and correctness
    X_train_matrix = tfidf_vectorizer.transform(X_train_text)
    X_test_matrix = tfidf_vectorizer.transform(X_test_text)
    
    # 3. Initialize and Train Model
    xgb_model = XGBClassifier(
        objective='binary:logistic', 
        n_estimators=100,             
        learning_rate=0.1,           
        random_state=RND,
        use_label_encoder=False,
        eval_metric='logloss',       
        n_jobs=-1                    
    )
    
    start_time = time.time()
    xgb_model.fit(X_train_matrix, y_train)
    training_time = time.time() - start_time
    
    # 4. Predict
    y_train_pred = xgb_model.predict(X_train_matrix) # For Overfitting Check
    y_test_pred = xgb_model.predict(X_test_matrix)   # For Test Score
    
    # 5. Evaluate
    # Use f1_score with 'weighted' average to get a single, stable comparison metric
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Calculate Overfitting Margin
    overfit_margin = train_f1 - test_f1
    
    # Calculate F1 for the Minority Class (False)
    # The f1_score function can return the score for each class if average=None
    f1_scores_per_class = f1_score(y_test, y_test_pred, average=None)
    # Assuming False is the 0 index
    minority_f1 = f1_scores_per_class[0] if len(f1_scores_per_class) > 1 else 'N/A'

    # Get sample support to assess imbalance
    support_false = y_test.value_counts().get(False, 0)
    support_true = y_test.value_counts().get(True, 0)
    
    # 6. Store Results
    result = {
        'Target': target_col,
        'Feature_Set': 'TF-IDF',
        'Train_F1_Weighted': round(train_f1, 4),
        'Test_F1_Weighted': round(test_f1, 4),
        'Overfit_Margin (Train - Test)': round(overfit_margin, 4),
        'Minority_Class_F1 (False)': round(minority_f1, 4) if isinstance(minority_f1, (int, float)) else minority_f1,
        'Imbalance_Support_False': support_false,
        'Imbalance_Support_True': support_true,
        'Training_Time_s': round(training_time, 2)
    }
    
    # Save the model and results
    all_tfidf_results.append(result)
    trained_tfidf_models[target_col] = xgb_model
    
    print(f"\n--- Results for **{target_col}** (TF-IDF) ---")
    print(f"Test F1 (Weighted): {test_f1:.4f}")
    print(f"Overfit Margin: {overfit_margin:.4f}")
    print(f"Minority Class F1 (False): {minority_f1:.4f}" if isinstance(minority_f1, (int, float)) else f"Minority Class F1: {minority_f1}")
    print("\nClassification Report (Test Set):")
    print(classification_report(y_test, y_test_pred))


# --- Execute the loop for all target columns ---
print("\n#####################################################")
print("### STARTING MULTI-LABEL TRAINING WITH TF-IDF ###")
print("#####################################################")

# Loop through all present target columns
for target in present_targets:
    train_and_evaluate_target(target, X, df)

# Convert results to a DataFrame for easy viewing
results_df_tfidf = pd.DataFrame(all_tfidf_results)

print("\n--- FINAL TF-IDF RESULTS SUMMARY ---")
print(results_df_tfidf)
print("\nAll models trained and results saved!")


#####################################################
### STARTING MULTI-LABEL TRAINING WITH TF-IDF ###
#####################################################


/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:09:42] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsToxic** (TF-IDF) ---
Test F1 (Weighted): 0.8453
Overfit Margin: 0.0323
Minority Class F1 (False): 0.3101

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.95      0.19      0.31       108
        True       0.88      1.00      0.94       643

    accuracy                           0.88       751
   macro avg       0.92      0.59      0.62       751
weighted avg       0.89      0.88      0.85       751



/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:09:46] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsAbusive** (TF-IDF) ---
Test F1 (Weighted): 0.7712
Overfit Margin: 0.0496
Minority Class F1 (False): 0.5963

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.95      0.43      0.60       260
        True       0.77      0.99      0.86       491

    accuracy                           0.80       751
   macro avg       0.86      0.71      0.73       751
weighted avg       0.83      0.80      0.77       751



...


--- Results for **IsObscene** (TF-IDF) ---
Test F1 (Weighted): 0.8571
Overfit Margin: 0.0441
Minority Class F1 (False): 0.9283

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.88      0.99      0.93       611
        True       0.86      0.40      0.55       140

    accuracy                           0.88       751
   macro avg       0.87      0.69      0.74       751
weighted avg       0.87      0.88      0.86       751



/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:09:55] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsHatespeech** (TF-IDF) ---
Test F1 (Weighted): 0.8340
Overfit Margin: 0.0319
Minority Class F1 (False): 0.9107

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.84      1.00      0.91       558
        True       0.98      0.45      0.61       193

    accuracy                           0.85       751
   macro avg       0.91      0.72      0.76       751
weighted avg       0.87      0.85      0.83       751



/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:09:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsRacist** (TF-IDF) ---
Test F1 (Weighted): 0.8413
Overfit Margin: 0.0394
Minority Class F1 (False): 0.9165

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.85      0.99      0.92       576
        True       0.94      0.43      0.59       175

    accuracy                           0.86       751
   macro avg       0.90      0.71      0.76       751
weighted avg       0.87      0.86      0.84       751


--- FINAL TF-IDF RESULTS SUMMARY ---
          Target Feature_Set  Train_F1_Weighted  Test_F1_Weighted  \
0        IsToxic      TF-IDF             0.8776            0.8453   
1      IsAbusive      TF-IDF             0.8208            0.7712   
2  IsProvocative      TF-IDF             0.8285            0.7462   
3      IsObscene      TF-IDF             0.9012            0.8571   
4   IsHatespeech      TF-IDF             0.8658            0.8340   
5       IsRacist      TF-IDF             0.8807            

# Word2Vec Feature Generation

## Generate Averaged Word2Vec Features

In [21]:
# Install gensim if you haven't already: pip install gensim
from gensim.models import Word2Vec
import numpy as np
from typing import List

# Assuming 'df' and 'tokens' column are available from previous steps

# 1. Train the Word2Vec Model
# We use the list of lists (df['tokens']) as the training corpus
vector_size = 100 # Standard size for embeddings (100 to 300 is common)
window = 5        # Max distance between current and predicted word
min_count = 5     # Ignore all words with total frequency lower than this

w2v_model = Word2Vec(
    sentences=df['tokens'],
    vector_size=vector_size,
    window=window,
    min_count=min_count,
    workers=4, # Use multiple cores for speed
    seed=RND
)

print('\n--- Word2Vec Model Training Summary ---')
print(f"Model trained with {len(w2v_model.wv.key_to_index)} unique words (vocabulary size).")
print(f"Each word is represented by a vector of size: {vector_size}")
print('-------------------------------------------')

# 2. Create the Averaging Function

def document_vector(word_list: List[str], model: Word2Vec, size: int) -> np.ndarray:
    """Averages the vectors of all words in a list."""
    
    # Get the keyed vectors component
    wv = model.wv 
    
    # Initialize a vector of zeros
    vec = np.zeros(size)
    count = 0.
    
    for word in word_list:
        if word in wv:
            vec += wv[word]
            count += 1
            
    if count != 0:
        # Divide the sum by the count to get the average
        vec /= count
        
    return vec

# 3. Apply the averaging function to create the feature matrix (X_w2v)
# The result will be a list of 100-dimensional NumPy arrays
X_w2v_list = df['tokens'].apply(lambda x: document_vector(x, w2v_model, vector_size))

# Stack the list of arrays into a single NumPy matrix
X_w2v = np.stack(X_w2v_list.values)

print(f"\nFinal Averaged Word2Vec Feature Matrix (X_w2v) shape: {X_w2v.shape}")


--- Word2Vec Model Training Summary ---
Model trained with 2488 unique words (vocabulary size).
Each word is represented by a vector of size: 100
-------------------------------------------

Final Averaged Word2Vec Feature Matrix (X_w2v) shape: (3751, 100)


🚀 Step 2: Training XGBoost with Word2Vec Features

In [22]:
# Assuming X_w2v and all necessary imports/definitions from the previous step are available
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from xgboost import XGBClassifier
import time

# List to store all new results
all_w2v_results = []
trained_w2v_models = {}

def train_and_evaluate_w2v(target_col, X_matrix, y_data):
    """Splits Word2Vec data, trains XGBoost, evaluates, and reports overfitting."""
    
    # 1. Split Data (Using the X_w2v matrix and the current target for stratification)
    y = y_data[target_col]
    X_train_w2v, X_test_w2v, y_train, y_test = train_test_split(
        X_matrix, y, 
        test_size=0.2, 
        random_state=RND, 
        stratify=y
    )
    
    # 2. Initialize and Train Model (Same parameters as before)
    xgb_model = XGBClassifier(
        objective='binary:logistic', 
        n_estimators=100,             
        learning_rate=0.1,           
        random_state=RND,
        use_label_encoder=False,
        eval_metric='logloss',       
        n_jobs=-1                    
    )
    
    start_time = time.time()
    xgb_model.fit(X_train_w2v, y_train)
    training_time = time.time() - start_time
    
    # 3. Predict and Evaluate
    y_train_pred = xgb_model.predict(X_train_w2v) 
    y_test_pred = xgb_model.predict(X_test_w2v)  
    
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    overfit_margin = train_f1 - test_f1
    
    f1_scores_per_class = f1_score(y_test, y_test_pred, average=None)
    minority_f1 = f1_scores_per_class[0] if len(f1_scores_per_class) > 1 else 'N/A'
    
    support_false = y_test.value_counts().get(False, 0)
    support_true = y_test.value_counts().get(True, 0)
    
    # 4. Store Results
    result = {
        'Target': target_col,
        'Feature_Set': 'Word2Vec',
        'Train_F1_Weighted': round(train_f1, 4),
        'Test_F1_Weighted': round(test_f1, 4),
        'Overfit_Margin (Train - Test)': round(overfit_margin, 4),
        'Minority_Class_F1 (False)': round(minority_f1, 4) if isinstance(minority_f1, (int, float)) else minority_f1,
        'Imbalance_Support_False': support_false,
        'Imbalance_Support_True': support_true,
        'Training_Time_s': round(training_time, 2)
    }
    
    all_w2v_results.append(result)
    trained_w2v_models[target_col] = xgb_model
    
    print(f"\n--- Results for **{target_col}** (Word2Vec) ---")
    print(f"Test F1 (Weighted): {test_f1:.4f}")
    print(f"Overfit Margin: {overfit_margin:.4f}")
    print("\nClassification Report (Test Set):")
    print(classification_report(y_test, y_test_pred))


# --- Execute the loop for all target columns ---
print("\n#####################################################")
print("### STARTING MULTI-LABEL TRAINING WITH WORD2VEC ###")
print("#####################################################")

for target in present_targets:
    train_and_evaluate_w2v(target, X_w2v, df)

# Convert results to a DataFrame for comparison
results_df_w2v = pd.DataFrame(all_w2v_results)


#####################################################
### STARTING MULTI-LABEL TRAINING WITH WORD2VEC ###
#####################################################


/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:18:38] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsToxic** (Word2Vec) ---
Test F1 (Weighted): 0.8270
Overfit Margin: 0.1535

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.68      0.14      0.23       108
        True       0.87      0.99      0.93       643

    accuracy                           0.87       751
   macro avg       0.78      0.56      0.58       751
weighted avg       0.85      0.87      0.83       751



/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:18:46] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsAbusive** (Word2Vec) ---
Test F1 (Weighted): 0.6892
Overfit Margin: 0.2838

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.63      0.39      0.48       260
        True       0.73      0.88      0.80       491

    accuracy                           0.71       751
   macro avg       0.68      0.64      0.64       751
weighted avg       0.70      0.71      0.69       751



...


--- Results for **IsObscene** (Word2Vec) ---
Test F1 (Weighted): 0.8205
Overfit Margin: 0.1704

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.85      1.00      0.92       611
        True       0.92      0.25      0.39       140

    accuracy                           0.86       751
   macro avg       0.89      0.62      0.66       751
weighted avg       0.87      0.86      0.82       751



/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:18:58] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsHatespeech** (Word2Vec) ---
Test F1 (Weighted): 0.7837
Overfit Margin: 0.2073

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.81      0.97      0.88       558
        True       0.81      0.35      0.49       193

    accuracy                           0.81       751
   macro avg       0.81      0.66      0.69       751
weighted avg       0.81      0.81      0.78       751



/workspaces/proyecto10-grupo5/venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:19:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- Results for **IsRacist** (Word2Vec) ---
Test F1 (Weighted): 0.8012
Overfit Margin: 0.1911

Classification Report (Test Set):
              precision    recall  f1-score   support

       False       0.83      0.97      0.90       576
        True       0.80      0.35      0.49       175

    accuracy                           0.83       751
   macro avg       0.82      0.66      0.69       751
weighted avg       0.82      0.83      0.80       751



Step 3: Compare Results

In [23]:
# Assuming results_df_tfidf and results_df_w2v were successfully created
comparison_df = pd.concat([results_df_tfidf, results_df_w2v], ignore_index=True)

# Select and format key comparison columns
comparison_summary = comparison_df[[
    'Target', 
    'Feature_Set', 
    'Test_F1_Weighted', 
    'Minority_Class_F1 (False)', 
    'Overfit_Margin (Train - Test)',
    'Training_Time_s'
]]

# Sort for easier comparison
comparison_summary = comparison_summary.sort_values(by=['Target', 'Feature_Set'])

print("\n#####################################################")
print("### FINAL MODEL COMPARISON (TF-IDF vs. WORD2VEC) ###")
print("#####################################################")
print(comparison_summary)
print("\n")

# Provide an overall conclusion on which feature set performed better
best_f1 = comparison_summary.loc[comparison_summary.groupby('Target')['Test_F1_Weighted'].idxmax()]
print("--- SUMMARY: BEST PERFORMING FEATURE SET PER TARGET (by Weighted F1) ---")
print(best_f1[['Target', 'Feature_Set', 'Test_F1_Weighted', 'Minority_Class_F1 (False)']])
print("\nConclusion: Word2Vec features are significantly lower dimensional (100 features) than TF-IDF (3909 features), making them much faster to train, but you must look at the F1 scores to determine which is more effective.")


#####################################################
### FINAL MODEL COMPARISON (TF-IDF vs. WORD2VEC) ###
#####################################################
           Target Feature_Set  Test_F1_Weighted  Minority_Class_F1 (False)  \
1       IsAbusive      TF-IDF            0.7712                     0.5963   
7       IsAbusive    Word2Vec            0.6892                     0.4834   
4    IsHatespeech      TF-IDF            0.8340                     0.9107   
10   IsHatespeech    Word2Vec            0.7837                     0.8849   
3       IsObscene      TF-IDF            0.8571                     0.9283   
9       IsObscene    Word2Vec            0.8205                     0.9184   
2   IsProvocative      TF-IDF            0.7462                     0.8664   
8   IsProvocative    Word2Vec            0.7289                     0.8536   
5        IsRacist      TF-IDF            0.8413                     0.9165   
11       IsRacist    Word2Vec            0.8012           

In [32]:
import pandas as pd

# --- 1. Combine and Prepare Data ---
# Assuming results_df_tfidf and results_df_w2v are available
comparison_df = pd.concat([results_df_tfidf, results_df_w2v], ignore_index=True)

# Select key columns for the summary
comparison_summary = comparison_df[[
    'Target', 
    'Feature_Set', 
    'Test_F1_Weighted', 
    'Minority_Class_F1 (False)', 
    'Overfit_Margin (Train - Test)',
    'Training_Time_s'
]].copy()

# Sort by Target for side-by-side comparison
comparison_summary = comparison_summary.sort_values(by=['Target', 'Feature_Set'], ascending=[True, False])

# Reset index for clean display
comparison_summary.reset_index(drop=True, inplace=True)


# --- 2. Styling Function for Technical Readability ---
def highlight_best_features(s):
    """
    Highlights the better performing feature set (higher Test F1 is better).
    Also highlights the worst Minority F1 (imbalance failure) and worst Overfit Margins.
    """
    if s.name in ['Test_F1_Weighted', 'Minority_Class_F1 (False)']:
        # Group by Target and find the index of the max value
        max_idx = comparison_summary.groupby('Target')[s.name].idxmax()
        is_max = pd.Series(False, index=s.index)
        is_max[max_idx] = True
        
        # Color the best scores (Higher is better)
        return ['background-color: lightgreen' if v else '' for v in is_max]
    
    elif s.name == 'Overfit_Margin (Train - Test)':
        # Highlight margins that are too high (Word2Vec)
        is_high_overfit = s > 0.10
        # Color high margins (Higher is worse)
        return ['background-color: salmon' if v else '' for v in is_high_overfit]

    elif s.name == 'Training_Time_s':
        # Highlight the fastest model (Word2Vec)
        min_idx = comparison_summary.groupby('Target')[s.name].idxmin()
        is_min = pd.Series(False, index=s.index)
        is_min[min_idx] = True
        # Color fastest time (Lower is better)
        return ['background-color: lightblue' if v else '' for v in is_min]
    
    return ['' for _ in s]

# --- 3. Generate Styled Table for Technical Audience ---

print("## 🤓 Technical Model Comparison: TF-IDF vs. Word2Vec")
print("---")
# Apply styling and formatting
styled_comparison = comparison_summary.style.apply(highlight_best_features, axis=0)\
    .format({
        'Test_F1_Weighted': "{:.4f}",
        'Minority_Class_F1 (False)': "{:.4f}",
        'Overfit_Margin (Train - Test)': "{:.4f}",
        'Training_Time_s': "{:.2f}s"
    })\
    .set_caption("Color Key: Green = Best Score; Salmon = High Overfitting; Blue = Fastest Training Time")

display(styled_comparison) # Use display() in notebook environment


# --- 4. Final Overview for Non-Technical Audience ---

print("\n\n## 🏆 Final Performance Overview (The Winner's Circle)")
print("---")

# Determine the overall winner for each target based on Test F1 Weighted
best_f1_results = comparison_summary.loc[comparison_summary.groupby('Target')['Test_F1_Weighted'].idxmax()]

# Check if TF-IDF won all targets
if (best_f1_results['Feature_Set'] == 'TF-IDF').all():
    print("### 🟢 THE CLEAR WINNER: TF-IDF FEATURES")
    print("> The **TF-IDF (Term Frequency-Inverse Document Frequency) feature set** proved to be the best choice for all six toxicity categories.")
    print("* **Why TF-IDF Won:** It accurately captures which words are most unique and relevant to a specific toxicity label, leading to more reliable predictions than Word2Vec.")
else:
    print("### 🟡 MIXED RESULTS (Feature Set Winner per Category)")


# Generate the non-technical summary table
print("\n### 🥇 Best Feature Set per Toxicity Category")

summary_table = best_f1_results[['Target', 'Feature_Set', 'Test_F1_Weighted', 'Minority_Class_F1 (False)']]
summary_table['Test_F1_Weighted'] = summary_table['Test_F1_Weighted'].map('{:.4f}'.format)
summary_table['Minority_Class_F1 (False)'] = summary_table['Minority_Class_F1 (False)'].map('{:.4f}'.format)

print(summary_table.to_markdown(index=False))

print("\n### ⚠️ Critical Imbalance Warning")
print(f"**Action Item:** Notice the low **Minority Class F1** for the **IsToxic** category ({best_f1_results[best_f1_results['Target'] == 'IsToxic']['Minority_Class_F1 (False)'].iloc[0]:.4f}). This model is struggling to correctly identify comments that are *not* toxic. The next crucial step is to **fix this class imbalance** in the winning TF-IDF models to make them reliable.")

## 🤓 Technical Model Comparison: TF-IDF vs. Word2Vec
---


,Target,Feature_Set,Test_F1_Weighted,Minority_Class_F1 (False),Overfit_Margin (Train - Test),Training_Time_s
0,IsAbusive,Word2Vec,0.6892,0.4834,0.2838,4.94s
1,IsAbusive,TF-IDF,0.7712,0.5963,0.0496,3.29s
2,IsHatespeech,Word2Vec,0.7837,0.8849,0.2073,2.57s
3,IsHatespeech,TF-IDF,0.8340,0.9107,0.0319,2.43s
4,IsObscene,Word2Vec,0.8205,0.9184,0.1704,3.66s
5,IsObscene,TF-IDF,0.8571,0.9283,0.0441,1.85s
6,IsProvocative,Word2Vec,0.7289,0.8536,0.2522,2.76s
7,IsProvocative,TF-IDF,0.7462,0.8664,0.0823,2.16s
8,IsRacist,Word2Vec,0.8012,0.8969,0.1911,2.11s
9,IsRacist,TF-IDF,0.8413,0.9165,0.0394,2.17s




## 🏆 Final Performance Overview (The Winner's Circle)
---
### 🟢 THE CLEAR WINNER: TF-IDF FEATURES
> The **TF-IDF (Term Frequency-Inverse Document Frequency) feature set** proved to be the best choice for all six toxicity categories.
* **Why TF-IDF Won:** It accurately captures which words are most unique and relevant to a specific toxicity label, leading to more reliable predictions than Word2Vec.

### 🥇 Best Feature Set per Toxicity Category
| Target        | Feature_Set   |   Test_F1_Weighted |   Minority_Class_F1 (False) |
|:--------------|:--------------|-------------------:|----------------------------:|
| IsAbusive     | TF-IDF        |             0.7712 |                      0.5963 |
| IsHatespeech  | TF-IDF        |             0.834  |                      0.9107 |
| IsObscene     | TF-IDF        |             0.8571 |                      0.9283 |
| IsProvocative | TF-IDF        |             0.7462 |                      0.8664 |
| IsRacist      | TF-IDF        |        

/tmp/ipykernel_4824/2222821343.py:93: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_table['Test_F1_Weighted'] = summary_table['Test_F1_Weighted'].map('{:.4f}'.format)
/tmp/ipykernel_4824/2222821343.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  summary_table['Minority_Class_F1 (False)'] = summary_table['Minority_Class_F1 (False)'].map('{:.4f}'.format)


In [26]:
!pip install tabulate

  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)


In [27]:
# Assuming the 'tabulate' installation failed or the kernel restart didn't help.
# We will use the standard pandas 'to_string' method instead of 'to_markdown'.

import pandas as pd

# --- Final Overview for Non-Technical Audience ---

print("\n\n## 🏆 Final Performance Overview (The Winner's Circle)")
print("---")

# Determine the overall winner for each target based on Test F1 Weighted
best_f1_results = comparison_summary.loc[comparison_summary.groupby('Target')['Test_F1_Weighted'].idxmax()]

# Check if TF-IDF won all targets
if (best_f1_results['Feature_Set'] == 'TF-IDF').all():
    print("### 🟢 THE CLEAR WINNER: TF-IDF FEATURES")
    print("> The **TF-IDF (Term Frequency-Inverse Document Frequency) feature set** proved to be the best choice for all six toxicity categories.")
    print("* **Why TF-IDF Won:** It accurately captures which words are most unique and relevant to a specific toxicity label, leading to more reliable predictions and less overfitting than Word2Vec.")
else:
    print("### 🟡 MIXED RESULTS (Feature Set Winner per Category)")


# Generate the non-technical summary table
print("\n### 🥇 Best Feature Set per Toxicity Category")

# Use .copy() to address the FutureWarning about incompatible dtype setting
summary_table = best_f1_results[['Target', 'Feature_Set', 'Test_F1_Weighted', 'Minority_Class_F1 (False)']].copy()

# Fix the incompatible dtype issue and format the columns
summary_table.loc[:, 'Test_F1_Weighted'] = summary_table['Test_F1_Weighted'].map('{:.4f}'.format)
summary_table.loc[:, 'Minority_Class_F1 (False)'] = summary_table['Minority_Class_F1 (False)'].map('{:.4f}'.format)

# Use to_string() instead of to_markdown() to avoid the 'tabulate' dependency
print(summary_table.to_string(index=False))

print("\n### ⚠️ Critical Imbalance Warning")
# Get the Minority F1 for IsToxic from the winning feature set
is_toxic_minority_f1 = best_f1_results[best_f1_results['Target'] == 'IsToxic']['Minority_Class_F1 (False)'].iloc[0]

print(f"**Action Item:** Notice the low **Minority Class F1** for the **IsToxic** category ({is_toxic_minority_f1:.4f}). This model is struggling to correctly identify comments that are *not* toxic. The next crucial step is to **fix this class imbalance** in the winning TF-IDF models to make them reliable.")



## 🏆 Final Performance Overview (The Winner's Circle)
---
### 🟢 THE CLEAR WINNER: TF-IDF FEATURES
> The **TF-IDF (Term Frequency-Inverse Document Frequency) feature set** proved to be the best choice for all six toxicity categories.
* **Why TF-IDF Won:** It accurately captures which words are most unique and relevant to a specific toxicity label, leading to more reliable predictions and less overfitting than Word2Vec.

### 🥇 Best Feature Set per Toxicity Category
       Target Feature_Set Test_F1_Weighted Minority_Class_F1 (False)
    IsAbusive      TF-IDF           0.7712                    0.5963
 IsHatespeech      TF-IDF           0.8340                    0.9107
    IsObscene      TF-IDF           0.8571                    0.9283
IsProvocative      TF-IDF           0.7462                    0.8664
     IsRacist      TF-IDF           0.8413                    0.9165
      IsToxic      TF-IDF           0.8453                    0.3101

### ⚠️ Critical Imbalance Warning
**Action Ite

/tmp/ipykernel_4824/2418830927.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['0.7712' '0.8340' '0.8571' '0.7462' '0.8413' '0.8453']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  summary_table.loc[:, 'Test_F1_Weighted'] = summary_table['Test_F1_Weighted'].map('{:.4f}'.format)
/tmp/ipykernel_4824/2418830927.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['0.5963' '0.9107' '0.9283' '0.8664' '0.9165' '0.3101']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  summary_table.loc[:, 'Minority_Class_F1 (False)'] = summary_table['Minority_Class_F1 (False)'].map('{:.4f}'.format)


📈 1. Average Overfitting Score Calculation

In [30]:
import pandas as pd

# Assuming comparison_df is available from the previous steps
# This DataFrame holds the results for both TF-IDF and Word2Vec

# Group the results by Feature_Set and calculate the mean of the Overfit_Margin
average_overfit_df = comparison_df.groupby('Feature_Set')['Overfit_Margin (Train - Test)'].mean().reset_index()
average_overfit_df.columns = ['Feature Set', 'Average Overfit Margin']

# Add interpretation based on the calculated values
def interpret_margin(margin):
    if margin < 0.10:
        return 'Excellent. Models are reliable and generalize well.'
    else:
        return 'Poor. Models are highly overfit and would perform poorly in production.'

average_overfit_df['Interpretation'] = average_overfit_df['Average Overfit Margin'].apply(interpret_margin)

# Reorder columns to match the requested output
average_overfit_df = average_overfit_df[['Feature Set', 'Average Overfit Margin', 'Interpretation']]

# Final Display
print("\n## 📊 Final Overfitting Scores Summary")
print("---")
print("Interpretation: Margin < 0.10 is excellent generalization; Margin > 0.10 suggests overfitting.")
print(average_overfit_df.to_markdown(index=False))


## 📊 Final Overfitting Scores Summary
---
Interpretation: Margin < 0.10 is excellent generalization; Margin > 0.10 suggests overfitting.
| Feature Set   |   Average Overfit Margin | Interpretation                                                          |
|:--------------|-------------------------:|:------------------------------------------------------------------------|
| TF-IDF        |                 0.0466   | Excellent. Models are reliable and generalize well.                     |
| Word2Vec      |                 0.209717 | Poor. Models are highly overfit and would perform poorly in production. |


💾 2. Saving Models

In [31]:
import joblib
import os
from xgboost import XGBClassifier # Needed to properly load models if used later

# --- Define Paths ---
# Notebook location: eda/04-understandingXGBoost.ipynb
# Target location: resources/models/
DEPLOYMENT_DIR = '../resources/models/'
os.makedirs(DEPLOYMENT_DIR, exist_ok=True) # Creates the directory if it doesn't exist

# --- 1. Save the TF-IDF Vectorizer ---
tfidf_vectorizer_path = os.path.join(DEPLOYMENT_DIR, 'tfidf_vectorizer.pkl')
# Assuming tfidf_vectorizer is available from the previous steps
joblib.dump(tfidf_vectorizer, tfidf_vectorizer_path)
print(f"✅ Saved TF-IDF Vectorizer to: {tfidf_vectorizer_path}")

# --- 2. Save the Winning XGBoost Models ---
# Assuming trained_tfidf_models (a dictionary of your 6 XGBoost models) is available
for target, model in trained_tfidf_models.items():
    model_path = os.path.join(DEPLOYMENT_DIR, f'xgb_tfidf_model_{target}.json')
    # Use the XGBoost recommended format (.json or .bin)
    model.save_model(model_path)
    print(f"✅ Saved XGBoost Model for {target} to: {model_path}")

print("\nAll feature extractors and models have been successfully saved for deployment!")

✅ Saved TF-IDF Vectorizer to: ../resources/models/tfidf_vectorizer.pkl
✅ Saved XGBoost Model for IsToxic to: ../resources/models/xgb_tfidf_model_IsToxic.json
✅ Saved XGBoost Model for IsAbusive to: ../resources/models/xgb_tfidf_model_IsAbusive.json
✅ Saved XGBoost Model for IsProvocative to: ../resources/models/xgb_tfidf_model_IsProvocative.json
✅ Saved XGBoost Model for IsObscene to: ../resources/models/xgb_tfidf_model_IsObscene.json
✅ Saved XGBoost Model for IsHatespeech to: ../resources/models/xgb_tfidf_model_IsHatespeech.json
✅ Saved XGBoost Model for IsRacist to: ../resources/models/xgb_tfidf_model_IsRacist.json

All feature extractors and models have been successfully saved for deployment!


- ✅ Saved TF-IDF Vectorizer to: ../resources/models/tfidf_vectorizer.pkl
- ✅ Saved XGBoost Model for IsToxic to: ../eda/json/xgb_tfidf_model_IsAbusive.json
- ✅ Saved XGBoost Model for IsAbusive to: ../eda/json/xgb_tfidf_model_IsAbusive.json
- ✅ Saved XGBoost Model for IsProvocative to: ../eda/json/xgb_tfidf_model_IsProvocative.json
- ✅ Saved XGBoost Model for IsObscene to: ../eda/json/xgb_tfidf_model_IsObscene.json
- ✅ Saved XGBoost Model for IsHatespeech to: ../eda/json/xgb_tfidf_model_IsHatespeech.json
- ✅ Saved XGBoost Model for IsRacist to: ../eda/json/xgb_tfidf_model_IsRacist.json

# 🚀 Multi-Label Toxicity Classification Pipeline Summary

## 1. Data Source and Objective
- **Dataset:** `synonym_youtoxic_english_1000.csv` (Total Samples: 3,751)
- **Objective:** Train separate XGBoost models for each of the six target columns to classify comments as toxic or non-toxic in six dimensions.
- **Target Labels:** `IsToxic`, `IsAbusive`, `IsProvocative`, `IsObscene`, `IsHatespeech`, `IsRacist`
- **Model:** XGBoost Classifier (XGBClassifier)

---

## 2. Text Preprocessing (Cleaning and Normalization)

The following sequence of steps was applied to the raw text to create the clean **`tokens`** list:

1.  **URL and Special Character Removal:** Used regular expressions (`re`) to remove web links and unusual symbols/emojis, preserving only basic punctuation.
2.  **Lowercasing:** Converted all text to uniform lowercase.
3.  **Tokenization and Lemmatization (spaCy):**
    * Used the **spaCy `en_core_web_sm`** model to break text into tokens and reduce words to their base dictionary form (lemma).
    * Removed spaCy's **default stop words** (e.g., 'the', 'a', 'is').
4.  **Custom Vocabulary Pruning (High-Frequency Filter):**
    * Calculated the global frequency of all remaining tokens.
    * The **top 100 most frequent words** (e.g., 'black', 'people', 'police') were removed to filter out domain-specific words that do not distinguish between toxic and non-toxic comments.

---

## 3. Feature Engineering and Comparison

Two distinct feature sets were generated and compared:

### A. TF-IDF Vectorization (Count-Based)
- **Method:** Created a high-dimensional, sparse matrix based on the frequency and distinctiveness of words in the corpus.
- **Output:** **3,909 features** (unigrams and bigrams).
- **Result:** **Winning Feature Set.** Demonstrated high predictive quality and **low overfitting** (Average Margin: **0.0466**).

### B. Averaged Word2Vec (Semantic-Based)
- **Method:** Trained a continuous Word2Vec embedding model (`vector_size=100`) and averaged all word vectors in a comment to create one dense document vector.
- **Output:** **100 features**.
- **Result:** **Losing Feature Set.** Showed signs of **severe overfitting** (Average Margin: **0.2097**), indicating poor generalization to unseen data.

---

## 4. Model Performance Summary (TF-IDF Winners)

| Target | Feature Set | Test F1 (Weighted) | Overfit Margin (Train - Test) | Key Issue |
| :--- | :--- | :--- | :--- | :--- |
| **IsToxic** | **TF-IDF** | 0.8453 | 0.0323 | **CRITICAL:** Minority F1 is **0.3101** (Severe Imbalance) |
| **IsAbusive** | **TF-IDF** | 0.7712 | 0.0496 | Minority F1 is 0.5963 (Imbalance) |
| **IsObscene** | **TF-IDF** | 0.8571 | 0.0441 | Well-generalized |
| **IsHatespeech** | **TF-IDF** | 0.8340 | 0.0319 | Well-generalized |
| **IsRacist** | **TF-IDF** | 0.8413 | 0.0394 | Well-generalized |
| **IsProvocative**| **TF-IDF** | 0.7462 | 0.0823 | Well-generalized |

---

## 5. Next Steps and Deployment

1.  **Model Selection:** The **TF-IDF features** will be used for the final application due to superior F1 scores and generalization.
2.  **Model Optimization:** The next step is to **address the class imbalance** (e.g., using `scale_pos_weight` in XGBoost) to significantly improve the prediction reliability for the minority class (especially `IsToxic`).
3.  **Deployment:** The following necessary components were saved to the `resources/models/` folder:
    * `tfidf_vectorizer.pkl` (The feature extractor)
    * Six `.json` files (e.g., `xgb_tfidf_model_IsToxic.json`) containing the trained XGBoost models.